# Data Collection and Processing

In [3]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import re
import os

### Load the data from zipfile

In [4]:
import zipfile
with zipfile.ZipFile('../raw_data/data.zip', 'r') as z:
    print(z.namelist())

['tweet-bot-data/', 'tweet-bot-data/bot/', 'tweet-bot-data/bot/tweets.csv', 'tweet-bot-data/bot/users.csv', 'tweet-bot-data/real/', 'tweet-bot-data/real/tweets.csv', 'tweet-bot-data/real/users.csv']


In [5]:
with zipfile.ZipFile('../raw_data/data.zip', 'r') as z:
    with z.open('tweet-bot-data/real/users.csv') as f:
        real_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/users.csv') as f:
        bot_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/real/tweets.csv') as f:
        real_tweets = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/tweets.csv') as f:
        bot_tweets = pd.read_csv(f, encoding='latin-1')

/tmp/ipykernel_261393/2605224422.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  real_tweets = pd.read_csv(f, encoding='latin-1')
/tmp/ipykernel_261393/2605224422.py:9: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bot_tweets = pd.read_csv(f, encoding='latin-1')


### Create relational database tables from original CSVs

In [6]:
# Load users
with zipfile.ZipFile('../raw_data/data.zip', 'r') as z:
    with z.open('tweet-bot-data/real/users.csv') as f:
        real_users = pd.read_csv(f, encoding='latin-1')
    with z.open('tweet-bot-data/bot/users.csv') as f:
        bot_users = pd.read_csv(f, encoding='latin-1')

real_users['bot_label'] = 0
bot_users['bot_label'] = 1
users_raw = pd.concat([real_users, bot_users], ignore_index=True)

# Build users table
user_cols = ['id', 'name', 'screen_name', 'statuses_count', 'followers_count',
             'friends_count', 'favourites_count', 'listed_count', 'verified',
             'location', 'lang', 'created_at', 'description', 'bot_label']
users = users_raw[user_cols].drop_duplicates(subset='id').copy()
users.rename(columns={'id': 'user_id'}, inplace=True)

# Build locations table
locations = users[['location']].drop_duplicates().dropna().reset_index(drop=True)
locations.insert(0, 'location_id', range(1, len(locations) + 1))
users = users.merge(locations, on='location', how='left')

users.to_csv('../processed_data/users.csv', index=False)
locations.to_csv('../processed_data/locations.csv', index=False)
print(f"Users: {users.shape}, Locations: {locations.shape}")

# Process tweets in chunks
tweet_id_counter = [1]
hashtag_id_counter = [1]
tweet_cols = ['id', 'user_id', 'text', 'retweet_count', 'reply_count',
              'favorite_count', 'num_hashtags', 'num_urls', 'num_mentions', 'created_at']

tweets_out = []
hashtags_out = []

def process_chunk(chunk, bot_label):
    chunk = chunk[tweet_cols].drop_duplicates(subset='id').copy()
    chunk.rename(columns={'id': 'tweet_id'}, inplace=True)
    chunk['tweet_id'] = range(tweet_id_counter[0], tweet_id_counter[0] + len(chunk))
    tweet_id_counter[0] += len(chunk)
    
    # Extract hashtags
    tags = (chunk[['tweet_id', 'text']]
        .assign(hashtag=chunk['text'].str.findall(r'#\w+'))
        .explode('hashtag')
        .dropna(subset=['hashtag'])
        .reset_index(drop=True)[['tweet_id', 'hashtag']])
    tags['hashtag'] = tags['hashtag'].str.lower()
    tags.insert(0, 'hashtag_id', range(hashtag_id_counter[0], hashtag_id_counter[0] + len(tags)))
    hashtag_id_counter[0] += len(tags)
    
    return chunk, tags  # chunk.drop(columns=['text']), tags

chunksize = 100000

with zipfile.ZipFile('../raw_data/data.zip', 'r') as z:
    for fname, bot_label in [
        ('tweet-bot-data/real/tweets.csv', 0),
        ('tweet-bot-data/bot/tweets.csv', 1)
    ]:
        with z.open(fname) as f:
            for chunk in pd.read_csv(f, encoding='latin-1', chunksize=chunksize):
                t, h = process_chunk(chunk, bot_label)
                tweets_out.append(t)
                hashtags_out.append(h)
                print(f"Processed chunk, tweet_id up to {tweet_id_counter[0]}")

print("Concatenating...")
tweets = pd.concat(tweets_out, ignore_index=True)
hashtags = pd.concat(hashtags_out, ignore_index=True)

tweets.to_csv('../processed_data/tweets.csv', index=False)
hashtags.to_csv('../processed_data/hashtags.csv', index=False)

print(f"Tweets: {tweets.shape}")
print(f"Hashtags: {hashtags.shape}")
print("Done")

Users: (4465, 15), Locations: (1938, 2)
Processed chunk, tweet_id up to 100001


/tmp/ipykernel_261393/878274268.py:63: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, encoding='latin-1', chunksize=chunksize):


Processed chunk, tweet_id up to 200001


/tmp/ipykernel_261393/878274268.py:63: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, encoding='latin-1', chunksize=chunksize):


Processed chunk, tweet_id up to 300001


...

Processed chunk, tweet_id up to 4039363
Processed chunk, tweet_id up to 4139363


/tmp/ipykernel_261393/878274268.py:63: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, encoding='latin-1', chunksize=chunksize):


Processed chunk, tweet_id up to 4239363


/tmp/ipykernel_261393/878274268.py:63: DtypeWarning: Columns (7,10) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(f, encoding='latin-1', chunksize=chunksize):


Processed chunk, tweet_id up to 4339363
Processed chunk, tweet_id up to 4439363
Processed chunk, tweet_id up to 4449397
Concatenating...
Tweets: (4449396, 10)
Hashtags: (813618, 3)
Done


In [7]:
# Drop rows where tweet_id is not numeric
tweets = tweets[pd.to_numeric(tweets['tweet_id'], errors='coerce').notna()]
tweets['tweet_id'] = tweets['tweet_id'].astype('int64')

# Save to parquet files with fastparquet
users.to_parquet('../processed_data/users.parquet', engine='fastparquet', index=False)
tweets.to_parquet('../processed_data/tweets.parquet', engine='fastparquet', index=False)
hashtags.to_parquet('../processed_data/hashtags.parquet', engine='fastparquet', index=False)
locations.to_parquet('../processed_data/locations.parquet', engine='fastparquet', index=False)

In [8]:
print(users.columns)
print(tweets.columns)
print(hashtags.columns)
print(locations.columns)

Index(['user_id', 'name', 'screen_name', 'statuses_count', 'followers_count',
       'friends_count', 'favourites_count', 'listed_count', 'verified',
       'location', 'lang', 'created_at', 'description', 'bot_label',
       'location_id'],
      dtype='object')
Index(['tweet_id', 'user_id', 'text', 'retweet_count', 'reply_count',
       'favorite_count', 'num_hashtags', 'num_urls', 'num_mentions',
       'created_at'],
      dtype='object')
Index(['hashtag_id', 'tweet_id', 'hashtag'], dtype='object')
Index(['location_id', 'location'], dtype='object')


In [9]:
print(users[['statuses_count','followers_count','friends_count','favourites_count','listed_count']].describe())
print(tweets[['retweet_count','reply_count','favorite_count','num_hashtags','num_urls','num_mentions']].describe())

       statuses_count  followers_count  friends_count  favourites_count  \
count     4465.000000      4465.000000    4465.000000       4465.000000   
mean     13441.175364      1480.154087     904.136394       3668.311086   
std      27995.744440     15262.107520    2097.145259      10389.210150   
min          3.000000         0.000000       0.000000          0.000000   
25%        438.000000       110.000000     138.000000         11.000000   
50%       3517.000000       324.000000     319.000000        601.000000   
75%      14328.000000       961.000000     750.000000       3372.000000   
max     399555.000000    986837.000000   46310.000000     313954.000000   

       listed_count  
count   4465.000000  
mean      16.047032  
std      139.729230  
min        0.000000  
25%        0.000000  
50%        2.000000  
75%        6.000000  
max     6166.000000  
       retweet_count  reply_count  favorite_count  num_hashtags      num_urls  \
count   4.449395e+06    4449395.0    4.449395